# Extrator de olerites do side ADP Brasil

Este programa acessa https://expert.hcm.adp.com/adpweb e extrai todos os seus olerites.
Grava-os em PDF e arquivo pickle com dados puros e processados para contabilidade.

Usa o Chromium com Selenium.

Uso na linha de comando:

```shell
usage: adpscrape.py [-h] [--adp-user ADP_USER] [--adp-password ADP_PASSWORD] [--start-year START_YEAR]
                    [--headless | --no-headless] [--pdf | --no-pdf] [--pickle | --no-pickle] [--output OUTPUT]
                    [-d | --debug | --no-debug]

options:
  -h, --help                     show this help message and exit
  --adp-user ADP_USER            ADP username (default: $ADP_USER environment variable)
  --adp-password ADP_PASSWORD    ADP password (default: $ADP_PASSWORD environment variable)
  --start-year START_YEAR        Começa a extração a partir deste ano (default: $ADP_SINCE ou ano passado)
  --headless, --no-headless      Executa o navegador (Chromium) sem janela nehuma (só que não funciona) (use --no-headless para
                                 ver o navegador trabalhando; default $HEADLESS ou falso)
  --excel, --no-excel            Salva planilha Excel com diversas visões (default: $EXCEL ou salva sim)
  --pdf, --no-pdf                Salva PDFs dos demonstrativos (use --no-pdf para não salvar; default: $PDF ou salva sim)
  --pickle, --no-pickle          Salva arquivo Pickle com tudo o que foi extraido e dados processados (default: $PICKLE ou
                                 salva sim)
  --output OUTPUT                Pasta para escrever resultados (default: $OUTPUT ou pasta atual)
  -d, --debug, --no-debug        Debug mode
```

Todos esses parâmetros podem também ser passados no arquivo `.env`. Veja exemplo em [.env.example](.env.example).

## Como usar em um notebook

In [ ]:
import os
import io
import pandas
import datetime

import adpscrape

In [ ]:
demonstrativos=adpscrape.main()

---

## Como usar sem o browser

Se o arquivo pickle já tiver sido extraído anteriormente, use-o assim:

In [ ]:
import pickle
import pandas

with open('demonstrativos.pkl','rb') as f:
    demonstrativos = pickle.load(f)

### Acesse 1 demonstrativo

In [ ]:
d = 0

# DataFrame contábil
demonstrativos[d].contabil

# Tabelas extraídas em HTML
demonstrativos[d].tables[5]

# O que mais tem no demonstrativo 0:
vars(demonstrativos[0]).keys()
# dict_keys(['url', 'codigo', 'tipo', 'competencia', 'pagamento', 'pdf', 'tables', 'contabil'])

### Gera DataFrame contábil com todos os demonstrativos

In [ ]:
import adpscrape

contabilidade = adpscrape.contabilidade_total(demonstrativos)

contabilidade

### Calcula valores líquidos depositados no banco

In [ ]:
(
    contabilidade
    .query("~demonstrativo.str.contains('(outros)')")
    .groupby(by=['pagamento','demonstrativo'])
    .agg(
        liquido=('valor',sum)
    )
    # .sort_index()
    # .reset_index(level=1)
    # .set_index('demonstrativo', append=True)
)

### Tipos de contas e onde aparecem

In [ ]:
(
    contabilidade
    .groupby(['conta','demonstrativo'])
    .first()
    .reset_index()
    # .query("~demonstrativo.str.contains('(outros)')")
    .sort_values(['demonstrativo','conta'])
    [['conta','demonstrativo']]
)

### Relatório detalhado com benefícios

Primeiro vamos classificar e categorizar cada conta

In [ ]:
import yaml
import types

categorias = types.SimpleNamespace(
    yaml.safe_load("""
        # Valores que me descontam para benefícios
        Co-participações:
            - 3179 VA - DESCONTO
            - 3781 Bradesco - Desconto Dependente
            - 3783 Bradesco - COPARTICIPAÇÃO TITULAR
            - 3784 BRADESCO - COPARTICIPAÇÃO DEPENDENT
            - 5321 Seguro de Vida Desc.Autorizado



        # Descontos do governo
        Impostos:
            - 5560 INSS
            - 9960 INSS 13Salário
            - 5500 IR Retido



        # Horas extras
        Horas extras:
            - 1606 Horas Extras 70%
            - 3257 Banco de Horas 70%
            - 1804 Adicional Noturno 40 %



        # Valores que são descontados do líquido mensal, mas que eu recebo de outros jeitos
        Desvios:
            - 5590 Prev. Privada Básica



        # Benefícios que recebo de outras formas, tangíveis e intangíveis
        Benefícios:
            - 3179a Vale                   # valor fabricado a partir de '3179 VA - DESCONTO'
            - 3913a Contrib.Patrocinadora  # valor fabricado a partir de '3913 Contrib.Patrocinadora'
            - 1330a FGTS                   # valor fabricado a partir de '1330 FGTS RecMes - Remun' e '1331 FGTS RecMes - 13Sal'
            - 999 Carro
            - 888 Plano de saúde
    """
    )
)

categorias

Vamos fabricar alguns valores que recebemos como benefícos ou de outras formas

In [ ]:
beneficios = pandas.concat(
    [
        # fabrica valores para '3179a Vale' a partir
        # de '3179 VA - DESCONTO'
        (
            contabilidade
            .query("conta=='3179 VA - DESCONTO'")
            .assign(
                conta='3179a Vale',
                valor=lambda table: table.valor * -10,
            )
        ),

        # fabrica valores para '3913a Contrib.Patrocinadora' a
        # partir de '3913 Contrib.Patrocinadora'
        (
            contabilidade
            .query("conta=='3913 Contrib.Patrocinadora'")
            .assign(
                demonstrativo=lambda table: table.demonstrativo.str.replace(' (outros)', ''),
                conta='3913a Contrib.Patrocinadora',
                valor=lambda table: table.valor * -1,
            )
        ),

        # fabrica valores para '1330a FGTS' a partir de
        # '1330 FGTS RecMes - Remun' e '1331 FGTS RecMes - 13Sal'
        (
            contabilidade
            .query("conta.isin(['1330 FGTS RecMes - Remun','1331 FGTS RecMes - 13Sal'])")
            .assign(
                demonstrativo=lambda table: table.demonstrativo.str.replace(' (outros)', ''),
                conta='1330a FGTS',
                valor=lambda table: table.valor * -1,
            )
        )
    ]
)

In [ ]:
(
    pandas.concat([contabilidade,beneficios])
    .query("~demonstrativo.str.contains('(outros)')")
    .assign(
        # Assinala categorias para cada conta
        contacat=lambda table: (
            table.conta.map(
                {
                    value: category
                    for category, values in vars(categorias).items()
                    for value in values
                }
            )
            .fillna('Bruto')
            .astype('category')
        )
    )
    .groupby(by=['pagamento','demonstrativo','contacat'])
    .agg(
        valor=('valor',sum)
    )

    # Transpose values from long to wide
    .unstack('contacat')
    .fillna(0)

    # Remove level of columns
    .pipe(lambda table: table.set_axis(table.columns.droplevel(0), axis=1))

    # Now compute useful user values
    .assign(
        **{
            'Depósito':    lambda table: table['Bruto'] + table['Horas extras'] + table['Desvios'] + table['Co-participações'] + table['Impostos'],
            'Bruto Total': lambda table: table['Bruto'] + table['Horas extras'] + (-table['Desvios']) + table['Benefícios']
        }
    )

    # Visão totalizada por mês
    .pipe(lambda table: table.set_axis(pandas.DatetimeIndex(table.index.droplevel(1))))
    .resample('ME').sum()

    ["Bruto Total,Bruto,Horas extras,Desvios,Co-participações,Impostos,Depósito".split(',')]

    .style.format('R$ {:,.2f}')
)

In [ ]:
liquido no banco = normal + impostos + desvios + copart
salario com benefícios (EBITDA?) = normal + (-desvios) + benef + copart

In [ ]:
(
    pandas.concat([contabilidade,beneficios])
    .query("~demonstrativo.str.contains('(outros)')")
    .assign(
        # Assinala categorias para cada conta
        contacat=lambda table: (
            table.conta.map(
                {
                    value: category
                    for category, values in vars(categorias).items()
                    for value in values
                }
            )
            .fillna('normal')
            .astype('category')
        )
    )
    .assign(
        # Converte desvios em positivos, pois não são descontos reais
        valor=lambda table: (table.query("contacat=='desvios'").valor * -1).combine_first(table.valor)
    )
    .groupby(by=['pagamento','contacat'])
    .agg(
        liquido=('valor',sum)
    )
)

---

    .query("~demonstrativo.str.contains('(outros)')")
    .groupby(by=['pagamento','demonstrativo'])
    
Colunas:
- bruto:
    - exclui 'outros'
    - soma de todos os positivos
    - soma negativa de todos os desvios
- liquido total:
    - bruto
    - beneficios
- liquido no banco:
    - exclui 'outros'
    - soma de tudo
- impostos:
    - soma de todos da categoria impostos